# Ejercicios ensembling
En este ejercicio vas a realizar prediciones sobre un dataset de ciudadanos indios diabéticos. Se trata de un problema de clasificación en el que intentaremos predecir 1 (diabético) 0 (no diabético).

### 1. Carga las librerias que consideres comunes al notebook

In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

### 2. Lee los datos de [esta direccion](https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv)
Los nombres de columnas son:
```Python
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
```

In [20]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']

df = pd.read_csv(url, names=names)

df

,preg,plas,pres,skin,test,mass,pedi,age,class
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


In [21]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   preg    768 non-null    int64  
 1   plas    768 non-null    int64  
 2   pres    768 non-null    int64  
 3   skin    768 non-null    int64  
 4   test    768 non-null    int64  
 5   mass    768 non-null    float64
 6   pedi    768 non-null    float64
 7   age     768 non-null    int64  
 8   class   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [22]:
df.describe()
# posiblemente haya que hacer escalado, según el modelo que usemos
# algunos son robustos
# la segunda comuna (media) es 30 veces mayor que la primera
# la std de test es mucho mayor que la media
# cuando la std es igual o mayor que la media ya te está diciendo que tienes un conjunto con mucha variabilidad

,preg,plas,pres,skin,test,mass,pedi,age,class
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [23]:
df.corr() # esta siempre está en pearson

,preg,plas,pres,skin,test,mass,pedi,age,class
preg,1.000000,0.129459,0.141282,-0.081672,-0.073535,0.017683,-0.033523,0.544341,0.221898
plas,0.129459,1.000000,0.152590,0.057328,0.331357,0.221071,0.137337,0.263514,0.466581
pres,0.141282,0.152590,1.000000,0.207371,0.088933,0.281805,0.041265,0.239528,0.065068
skin,-0.081672,0.057328,0.207371,1.000000,0.436783,0.392573,0.183928,-0.113970,0.074752
test,-0.073535,0.331357,0.088933,0.436783,1.000000,0.197859,0.185071,-0.042163,0.130548
mass,0.017683,0.221071,0.281805,0.392573,0.197859,1.000000,0.140647,0.036242,0.292695
pedi,-0.033523,0.137337,0.041265,0.183928,0.185071,0.140647,1.000000,0.033561,0.173844
age,0.544341,0.263514,0.239528,-0.113970,-0.042163,0.036242,0.033561,1.000000,0.238356
class,0.221898,0.466581,0.065068,0.074752,0.130548,0.292695,0.173844,0.238356,1.000000


### 3. Bagging
Para este apartado tendrás que crear un ensemble utilizando la técnica de bagging ([BaggingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html)), mediante la cual combinarás 100 [DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html). Recuerda utilizar también [cross validation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html) con 10 kfolds.

**Para este apartado y siguientes, no hace falta que dividas en train/test**, por hacerlo más sencillo. Simplemente divide tus datos en features y target.

Establece una semilla

In [24]:
X = df.drop('class', axis=1)
y = df['class']

# división inicial para los primeros apartados
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [25]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

estimator = DecisionTreeClassifier(max_depth=3, random_state=42)

# esto crea el ensemble con bagging:
bag_clf = BaggingClassifier(
    estimator = estimator,
    n_estimators=100, # Cantidad de árboles
    max_samples=100, # Muestras utilizadas en bootstrapping (máximo 100 muestras)
    bootstrap=True, # Usamos bootstrapping: muestreo con reemplazo (los datos vuelven al conjunto)
    max_features = 3, # Features que utiliza en el bootstrapping. Cuanto más bajo, mejor generalizará y menos overfitting (cada árbol utilizará 3 features)
    random_state=42)

In [26]:
from sklearn.model_selection import cross_val_score

# esto es cross-validation:
bag_scores = cross_val_score(bag_clf, X, y, cv=10)
print(bag_scores)
print(bag_scores.mean())

[0.76623377 0.75324675 0.77922078 0.7012987  0.74025974 0.72727273
 0.77922078 0.76623377 0.71052632 0.80263158]
0.7526144907723855


In [27]:
# no es terrible, sale bastante uniforme
# F1 entre 0.7-0.8

### 4. Random Forest
En este caso entrena un [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) con 100 árboles y un `max_features` de 3. También con validación cruzada

In [28]:
from sklearn.ensemble import RandomForestClassifier

rnd_clf = RandomForestClassifier(n_estimators=100,
                                 max_features=3,
                                 max_leaf_nodes=16, # máximo número de hojas (nodos finales)
                                 random_state=42)

rnd_scores = cross_val_score(rnd_clf, X, y, cv=10)
print(rnd_scores)
print(rnd_scores.mean())
print(rnd_scores.std()) # para ver la desviación estándar


[0.76623377 0.77922078 0.76623377 0.66233766 0.7012987  0.79220779
 0.80519481 0.84415584 0.72368421 0.80263158]
0.7643198906356801
0.05151963183325334


In [29]:
# e sun poquito mejor, pero xq el criterio de parada lo hemos agrandado
# vemos que hay 0.66 y 0.84, es modelo más sensible que el anterior a os outliers
# igual es mas sensible a que haya datos raros, puede haber overfit

In [30]:
rnd_clf.fit(X, y)
rnd_clf.feature_importances_
# dos destacan, la segunda y la priera de la segunda fila, pero todas aportan algo
# a la vista de esto no quitaria features asi de primeras

array([0.04162784, 0.41565912, 0.04090018, 0.02968155, 0.04697549,
       0.20967337, 0.07697495, 0.13850751])

In [31]:
'''
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

log_clf = LogisticRegression(max_iter=1000, random_state=42)
tree_clf = DecisionTreeClassifier(random_state=42)
svm_clf = SVC(probability=True, random_state=42)

voting_clf = VotingClassifier(
    estimators=[('lr', log_clf), ('dt', tree_clf), ('svc', svm_clf)],
    voting='soft'
)

voting_clf.fit(X_train, y_train)
'''

"\nfrom sklearn.ensemble import VotingClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.svm import SVC\nfrom sklearn.tree import DecisionTreeClassifier\n\nlog_clf = LogisticRegression(max_iter=1000, random_state=42)\ntree_clf = DecisionTreeClassifier(random_state=42)\nsvm_clf = SVC(probability=True, random_state=42)\n\nvoting_clf = VotingClassifier(\n    estimators=[('lr', log_clf), ('dt', tree_clf), ('svc', svm_clf)],\n    voting='soft'\n)\n\nvoting_clf.fit(X_train, y_train)\n"

### 5. AdaBoost
Implementa un [AdaBoostClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html) con 30 árboles.

In [32]:
from sklearn.ensemble import AdaBoostClassifier

estimator = DecisionTreeClassifier(max_depth=1)

ada_clf = AdaBoostClassifier(estimator = estimator, # DECISSION TREE
                             n_estimators=30,
                             learning_rate=0.5, # CUÁNTO QUIERO QUE EL ÁRBOL INTENTE MEJORAR AL ANTERIOR
                             random_state=42)

ada_scores = cross_val_score(ada_clf, X, y, cv=10)
print(ada_scores)
print(ada_scores.mean())

[0.72727273 0.79220779 0.77922078 0.7012987  0.74025974 0.76623377
 0.79220779 0.80519481 0.76315789 0.81578947]
0.7682843472317157


In [33]:
ada_clf.fit(X, y)
ada_clf.feature_importances_

# como los nodelos están de acuerdo en las mismas feature importantes,
# podría ser interesate probar un modelo que solo tuviera esas 3

array([0.04385832, 0.41928559, 0.        , 0.        , 0.        ,
       0.30849116, 0.06179297, 0.16657197])

In [34]:
'''
from sklearn.ensemble import BaggingClassifier

bag_clf = BaggingClassifier(
    DecisionTreeClassifier(random_state=42),
    n_estimators=500,
    max_samples=100,
    bootstrap=True,
    random_state=42
)
bag_clf.fit(X_train, y_train)
'''

'\nfrom sklearn.ensemble import BaggingClassifier\n\nbag_clf = BaggingClassifier(\n    DecisionTreeClassifier(random_state=42),\n    n_estimators=500,\n    max_samples=100,\n    bootstrap=True,\n    random_state=42\n)\nbag_clf.fit(X_train, y_train)\n'

### 6. GradientBoosting
Implementa un [GradientBoostingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html) con 100 estimadores

In [35]:
from sklearn.ensemble import GradientBoostingClassifier

gbct = GradientBoostingClassifier(max_depth=2,
                                 n_estimators=100,
                                 learning_rate=1.0, # igual habría que bajarlo porque hemos puesto 100 arboles sencillos
                                 random_state=42)

gbct_scores = cross_val_score(gbct, X, y, cv=10)
print(gbct_scores)
print(gbct_scores.mean())

[0.74025974 0.64935065 0.68831169 0.61038961 0.7012987  0.72727273
 0.71428571 0.74025974 0.72368421 0.75      ]
0.7045112781954888


In [36]:
gbct.fit(X, y)
gbct.feature_importances_

# sigue siendo bastante similar

array([0.05025817, 0.39199791, 0.04653611, 0.02929512, 0.04523053,
       0.17218874, 0.138069  , 0.12642441])

In [37]:
'''from sklearn.ensemble import RandomForestClassifier

rnd_clf = RandomForestClassifier(n_estimators=500, max_leaf_nodes=16, random_state=42)
rnd_clf.fit(X_train, y_train)'''

'from sklearn.ensemble import RandomForestClassifier\n\nrnd_clf = RandomForestClassifier(n_estimators=500, max_leaf_nodes=16, random_state=42)\nrnd_clf.fit(X_train, y_train)'

### 7. XGBoost
Para este apartado utiliza un [XGBoostClassifier](https://docs.getml.com/latest/api/getml.predictors.XGBoostClassifier.html) con 100 estimadores. XGBoost no forma parte de la suite de modelos de sklearn, por lo que tendrás que instalarlo con pip install

In [38]:
# %pip install xgboost
import xgboost as xgb

xgb_clas = xgb.XGBClassifier(n_estimators=100, random_state=42)
# 100 estimadores con 700 y pico registros es excesivo

xgb_scores = cross_val_score(xgb_clas, X, y, cv=10)
print(xgb_scores)
print(xgb_scores.mean())

[0.68831169 0.74025974 0.75324675 0.67532468 0.74025974 0.75324675
 0.75324675 0.77922078 0.67105263 0.76315789]
0.7317327409432673


In [39]:
# no es el mejor modelo

In [40]:
'''
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier

ada_clf = AdaBoostClassifier(
    DecisionTreeClassifier(max_depth=1), n_estimators=200,
    learning_rate=0.5, random_state=42
)
ada_clf.fit(X_train, y_train)

gb_clf = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb_clf.fit(X_train, y_train)
'''

'\nfrom sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier\n\nada_clf = AdaBoostClassifier(\n    DecisionTreeClassifier(max_depth=1), n_estimators=200,\n    learning_rate=0.5, random_state=42\n)\nada_clf.fit(X_train, y_train)\n\ngb_clf = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)\ngb_clf.fit(X_train, y_train)\n'

### 8. Primeros resultados
Crea un dataframe con los resultados y sus algoritmos, ordenándolos de mayor a menor

In [41]:
resultados = pd.DataFrame({
    'Algoritmo': ['Bagging', 'Random Forest', 'AdaBoost', 'GradientBoosting', 'XGBoost'],
    'Accuracy': [bag_scores.mean(), rnd_scores.mean(), ada_scores.mean(), gbct_scores.mean(), xgb_scores.mean()]
})

resultados = resultados.sort_values('F1', ascending=False).reset_index(drop=True)
print(resultados)

KeyError: 'F1'

In [ ]:
'''
modelos = [log_clf, tree_clf, svm_clf, voting_clf, bag_clf, rnd_clf, ada_clf, gb_clf]
nombres = ['Logistic Regression', 'Decision Tree', 'SVC', 'Voting', 'Bagging', 'Random Forest', 'AdaBoost', 'GradientBoost']

resultados = []
for name, clf in zip(nombres, modelos):
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    resultados.append({'Algoritmo': name, 'Accuracy': accuracy_score(y_test, y_pred)})

df_res = pd.DataFrame(resultados).sort_values(by='Accuracy', ascending=False)
print(df_res)
'''

             Algoritmo  Accuracy
3               Voting  0.779221
6             AdaBoost  0.772727
5        Random Forest  0.766234
2                  SVC  0.766234
4              Bagging  0.759740
1        Decision Tree  0.746753
0  Logistic Regression  0.746753
7        GradientBoost  0.740260


### 9. Hiperparametrización
Vuelve a entrenar los modelos de nuevo, pero esta vez dividiendo el conjunto de datos en train/test y utilizando un gridsearch para encontrar los mejores hiperparámetros.

1. Entreno los modelos con train/test y busco los mejores accuracy score (con los mismos hiperparámetros que antes):

In [42]:
from sklearn.model_selection import train_test_split

X = df[['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age']]
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size = 0.2,
                                                    random_state=42)

In [43]:
from sklearn.metrics import accuracy_score

bag_clf.fit(X_train, y_train)
bag_y_pred = bag_clf.predict(X_test)
accuracy_score(y_test, bag_y_pred)

0.7532467532467533

In [44]:
rnd_clf.fit(X_train, y_train)
rnd_y_pred = rnd_clf.predict(X_test)
accuracy_score(y_test, rnd_y_pred)

0.7597402597402597

In [45]:
ada_clf.fit(X_train, y_train)
ada_y_pred = ada_clf.predict(X_test)
accuracy_score(y_test, ada_y_pred)

0.7727272727272727

In [46]:
gbct.fit(X_train, y_train)
gbct_y_pred = gbct.predict(X_test)
accuracy_score(y_test, gbct_y_pred)

0.7142857142857143

In [47]:
xgb_clas.fit(X_train, y_train)
xgb_y_pred = xgb_clas.predict(X_test)
accuracy_score(y_test, xgb_y_pred)

0.7207792207792207

In [49]:
resultados = pd.DataFrame({
    'Algoritmo': ['Bagging', 'Random Forest', 'AdaBoost', 'GradientBoosting', 'XGBoost'],
    'Accuracy': [accuracy_score(y_test, bag_y_pred), 
                 accuracy_score(y_test, rnd_y_pred),
                 accuracy_score(y_test, ada_y_pred),
                 accuracy_score(y_test, gbct_y_pred),
                 accuracy_score(y_test, xgb_y_pred)
                 ]
})

resultados2 = resultados.sort_values('Accuracy', ascending=False).reset_index(drop=True)
print(resultados2)

          Algoritmo  Accuracy
0          AdaBoost  0.772727
1     Random Forest  0.759740
2           Bagging  0.753247
3           XGBoost  0.720779
4  GradientBoosting  0.714286


In [ ]:
# mismos resultados que antes, sigue ganando AdaBoost

2. GridSearch (hechos con IA):

In [50]:
from sklearn.model_selection import GridSearchCV

param_grid_bagging = {
    'n_estimators': [50, 100, 200],
    'max_samples': [0.5, 0.7, 1.0],
    'max_features': [0.5, 0.7, 1.0],
}

bagging_base = BaggingClassifier(estimator=DecisionTreeClassifier(random_state=42), random_state=42)

grid_bagging = GridSearchCV(bagging_base, param_grid_bagging, cv=5, scoring='accuracy', n_jobs=-1)
grid_bagging.fit(X_train, y_train)

print("Mejores parámetros:", grid_bagging.best_params_)
print("Mejor accuracy CV:", grid_bagging.best_score_)
print("Accuracy Test:", grid_bagging.best_estimator_.score(X_test, y_test))

Mejores parámetros: {'max_features': 1.0, 'max_samples': 0.5, 'n_estimators': 50}
Mejor accuracy CV: 0.7866986538717845
Accuracy Test: 0.7532467532467533


In [ ]:
# Mejor Accuracy CV (best_score_)
# Es el accuracy obtenido durante el GridSearch, es decir, sobre los datos de entrenamiento usando cross validation (5 folds). Es el promedio de los 5 folds internos.

#Accuracy Test (.score(X_test, y_test))
# Es el accuracy obtenido al evaluar el mejor modelo encontrado sobre el conjunto de test, datos que el modelo nunca ha visto.

# Lo ideal es que ambos sean parecidos. Si el CV es mucho mayor que el Test, el modelo está haciendo overfitting.

In [ ]:
# A PARTIR DE AQUÍ ES IA Y DAVID NO ESTÁ MUY DE ACUERDO
# DA MAS COMPLEJIDAD CON FEATURES Y REGISTROS

In [51]:
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_features': [2, 3, 4], # aqui coge numero enteros y en el anterior porcentajes
    'max_leaf_nodes': [8, 16, 32, None], # para la complejidad de mos datos esto es excesivo
    'max_depth': [4, 6, 8, None]
}

rf_base = RandomForestClassifier(random_state=42)

grid_rf = GridSearchCV(rf_base, param_grid_rf, cv=5, scoring='accuracy', n_jobs=-1)
grid_rf.fit(X_train, y_train)

print("Mejores parámetros:", grid_rf.best_params_)
print("Mejor accuracy CV:", grid_rf.best_score_)
print("Accuracy Test:", grid_rf.best_estimator_.score(X_test, y_test))

# los mejores parámetros apuntan a overfit

Mejores parámetros: {'max_depth': 8, 'max_features': 4, 'max_leaf_nodes': 32, 'n_estimators': 200}
Mejor accuracy CV: 0.7948020791683327
Accuracy Test: 0.7597402597402597


In [52]:
param_grid_ada = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.5, 1.0]
}

ada_base = AdaBoostClassifier(random_state=42)

grid_ada = GridSearchCV(ada_base, param_grid_ada, cv=5, scoring='accuracy', n_jobs=-1)
grid_ada.fit(X_train, y_train)

print("Mejores parámetros:", grid_ada.best_params_)
print("Mejor accuracy CV:", grid_ada.best_score_)
print("Accuracy Test:", grid_ada.best_estimator_.score(X_test, y_test))

Mejores parámetros: {'learning_rate': 1.0, 'n_estimators': 100}
Mejor accuracy CV: 0.7801546048247368
Accuracy Test: 0.7402597402597403


In [ ]:
param_grid_gb = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3],
    'subsample': [0.7, 0.8, 1.0]
}

gb_base = GradientBoostingClassifier(random_state=42)

grid_gb = GridSearchCV(gb_base, param_grid_gb, cv=5, scoring='accuracy', n_jobs=-1)
grid_gb.fit(X_train, y_train)

print("Mejores parámetros:", grid_gb.best_params_)
print("Mejor accuracy CV:", grid_gb.best_score_)
print("Accuracy Test:", grid_gb.best_estimator_.score(X_test, y_test))

In [ ]:
param_grid_xgb = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3],
    'subsample': [0.7, 0.8, 1.0]
}

xgb_base = xgb.XGBClassifier(random_state=42)

grid_xgb = GridSearchCV(xgb_base, param_grid_xgb, cv=5, scoring='accuracy', n_jobs=-1)
grid_xgb.fit(X_train, y_train)

print("Mejores parámetros:", grid_xgb.best_params_)
print("Mejor accuracy CV:", grid_xgb.best_score_)
print("Accuracy Test:", grid_xgb.best_estimator_.score(X_test, y_test))

In [ ]:
resultados_grid = pd.DataFrame({
    'Algoritmo': ['BaggingClassifier', 'RandomForestClassifier', 'XGBoostClassifier', 
                  'AdaBoostClassifier', 'GradientBoostingClassifier'],
    'Accuracy Test': [
        grid_bagging.best_estimator_.score(X_test, y_test),
        grid_rf.best_estimator_.score(X_test, y_test),
        grid_xgb.best_estimator_.score(X_test, y_test),
        grid_ada.best_estimator_.score(X_test, y_test),
        grid_gb.best_estimator_.score(X_test, y_test)
    ]
})

resultados_grid = resultados_grid.sort_values('Accuracy Test', ascending=False).reset_index(drop=True)
print(resultados_grid)

In [ ]:
'''
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 500],
    'max_depth': [3, 5, 10],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

print(f"Mejores parámetros: {grid_search.best_params_}")
print(f"Mejor accuracy: {grid_search.best_score_}")
'''

Mejores parámetros: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 500}
Mejor accuracy: 0.7785152605624417


### 10. Conclusiones finales

Si no hacemos GridSearch, el mejor algoritmo es AdaBoost y el peor es GradientBoosting, quedando Random Forest en segundo lugar. Sin embargo, al realizar el GridSearch, AdaBoost cae al último lugar, GradientBoosting sube al primero, y Random Forest se mantiene como segundo.

Es curioso que AdaBoost consigue un accuracy de 0.77 - el mayor de todas las opciones probadas - haciéndolo con los hiperparámetros elegidos al principio, y baja a un 0.74 con el GridSearch. Yo optaría por usar este método con los hiperparámetros elegidos al principio, seguido de RandomForest, que mantiene buenas métricas durante todos los tests.

In [ ]:
# lo siento, he usado IA para esto porque estoy cansada :(

# Eficacia de los Ensembles: El modelo de Voting ha obtenido el mejor rendimiento inicial (Accuracy: 0.779)
# Esto demuestra que combinar las predicciones de modelos con naturalezas distintas (Regresión Logística, Árboles y SVM)
# ayuda a compensar los errores individuales de cada uno, logrando una mayor robustez

# Boosting vs. Bagging: En este caso particular, AdaBoost (0.772) ha superado ligeramente a los métodos basados en
# Bagging y Random Forest. El Boosting, al centrarse en corregir secuencialmente los errores de los modelos anteriores,
# parece haber capturado mejor los patrones de este dataset. Curiosamente, el GradientBoost estándar obtuvo un resultado
# más bajo (0.740), lo que sugiere que sin optimizar parámetros puede caer en sobreajuste

# Modelos Simples vs. Complejos: Los modelos individuales como el Decision Tree o la Logistic Regression se encuentran
# en la parte baja de la tabla (0.746). Esto confirma la hipótesis principal del ejercicio: los métodos de ensamblado
# suelen mejorar la capacidad de generalización respecto a los modelos base

# Optimización mediante GridSearch: Tras realizar la búsqueda de hiperparámetros para el Random Forest, hemos encontrado
# que la combinación óptima es max_depth: 10, min_samples_split: 5 y n_estimators: 500. El Accuracy obtenido (0.778) es
# muy similar al del Voting inicial, lo que indica que un solo algoritmo bien tunneado puede ser tan potente como un
# conjunto de varios modelos distintos

# Reflexión final: Para este problema de clasificación de diabetes, si buscamos la máxima precisión y estabilidad,
# el Voting Classifier o un Random Forest optimizado serían las mejores elecciones